### Silver - Clientes

Problemas identificados:
- `customer_id` em caixa baixa (`c0051`)
- 3 registros com duplicidade de conteúdo:
  - `C0010`: a segunda versão preenche o campo `segmento` (Financeiro),
    anteriormente nulo
  - `C0025`: a segunda versão apresenta estado divergente (RJ vs Sta
    Catarina) — conflito de conteúdo, não apenas de formatação
  - `C0051`: a segunda versão contém e-mail inválido, sem "@"
- `estado` com mistura de sigla, nome completo, abreviação e variação de
  caixa
- `data_cadastro` em 3 formatos
- `status_cliente` com `Ativo`/`ATIVO`/`ativo`/`inativo`/null

As duplicidades são resolvidas pela versão com `updated_at` mais recente.
Para `C0051`, as duas versões possuem `updated_at` idêntico — não há
"mais recente" real nesse caso. Sem um critério de desempate adicional, o
resultado dependeria do particionamento físico dos dados no cluster, que
não é determinístico em Spark distribuído. E-mail válido é utilizado como
segundo critério de ordenação, garantindo resultado consistente entre
execuções.

Para `C0025` (conflito de estado), o mesmo critério é aplicado, mas a
divergência é sinalizada separadamente ao final do notebook — trata-se de
uma divergência de conteúdo que recomenda confirmação com a área de CRM.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_clientes_bronze = spark.table(f"{schema_name}.bronze_clientes")
df_map_uf = spark.table(f"{schema_name}.apoio_map_uf")

def normalizar_texto(col):
    c = F.upper(F.trim(col))
    return F.translate(c, "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ", "AAAAAEEEEIIIIOOOOOUUUUC")

Utilização de try_to_date/try_to_timestamp em vez das funções padrão, em
razão do modo ANSI do runtime (ver notebook de vendedores) — sem essa
substituição, o coalesce interrompe a execução na primeira tentativa de
formato incompatível, em vez de avaliar a próxima.

In [ ]:
df_clientes_step1 = (
    df_clientes_bronze
    .withColumn("customer_id", F.upper(F.trim(F.col("customer_id"))))
    .withColumn("updated_at_ts", F.expr("try_to_timestamp(updated_at)"))
)

In [ ]:
email_valido_expr = F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

w_dedup = Window.partitionBy("customer_id").orderBy(
    F.col("updated_at_ts").desc(),
    email_valido_expr.desc(),
)

df_clientes_dedup = (
    df_clientes_step1
    .withColumn("rk", F.row_number().over(w_dedup))
    .filter(F.col("rk") == 1)
    .drop("rk")
)

In [ ]:
df_clientes_uf = (
    df_clientes_dedup
    .withColumn("estado_norm", normalizar_texto(F.col("estado")))
    .join(df_map_uf, F.col("estado_norm") == F.col("variacao_norm"), "left")
)

In [ ]:
df_clientes_silver = (
    df_clientes_uf
    .withColumn(
        "data_cadastro_parsed",
        F.coalesce(
            F.expr("try_to_date(data_cadastro, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_cadastro, 'yyyy/MM/dd')"),
            F.expr("try_to_date(data_cadastro, 'dd/MM/yyyy')"),
        )
    )
    .withColumn(
        "status_cliente_norm",
        F.when(F.lower(F.trim(F.col("status_cliente"))) == "ativo", "Ativo")
         .when(F.lower(F.trim(F.col("status_cliente"))) == "inativo", "Inativo")
         .otherwise("Não informado")
    )
    .withColumn("email_valido", email_valido_expr)
    .withColumn("segmento_norm", F.coalesce(F.col("segmento"), F.lit("Não informado")))
    .withColumn("porte_norm", F.coalesce(F.initcap(F.trim(F.col("porte"))), F.lit("Não informado")))
    .select(
        F.col("customer_id"),
        F.col("nome_cliente"),
        F.col("segmento_norm").alias("segmento"),
        F.col("porte_norm").alias("porte"),
        F.col("cidade"),
        F.col("uf_sigla").alias("estado_uf"),
        F.col("regiao_geografica"),
        F.col("data_cadastro_parsed").alias("data_cadastro"),
        F.col("email"),
        F.col("email_valido"),
        F.col("status_cliente_norm").alias("status_cliente"),
        F.col("updated_at_ts").alias("updated_at"),
    )
)

(
    df_clientes_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_clientes")
)

print("bronze:", df_clientes_bronze.count(), "-> silver:", df_clientes_silver.count())

In [ ]:
print("E-mails inválidos:", df_clientes_silver.filter(~F.col("email_valido")).count())
print()
print("Observação: o cliente C0025 apresentava estado divergente entre as 2")
print("versões do cadastro (RJ vs Sta Catarina). Mantida a versão mais recente;")
print("recomenda-se confirmação com a área de CRM.")